# Classification Tutorial - DeepTab v2.0

This notebook demonstrates how to train classification models with DeepTab using the sklearn-compatible API.

**Topics covered:**
- Basic classification workflow
- Customization with configs (ModelConfig, PreprocessingConfig, TrainerConfig)
- Handling class imbalance with stratified splits
- Integration with scikit-learn (GridSearchCV, cross-validation)
- Advanced patterns (mixed data types, embeddings, ensembles)

**Requirements:**
```bash
pip install deeptab scikit-learn pandas numpy matplotlib
```

## 1. Setup and Data Generation

Import libraries and generate synthetic classification data.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from deeptab.models import MambularClassifier

# Set random seed for reproducibility
np.random.seed(42)

# Generate synthetic data
n_samples, n_features = 1000, 5
X = np.random.randn(n_samples, n_features)
y_continuous = np.dot(X, np.random.randn(n_features)) + np.random.randn(n_samples)

# Create DataFrame with numerical features
df = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(n_features)])

# Convert to multiclass classification (4 classes)
df["target"] = pd.qcut(y_continuous, q=4, labels=False)

print(f"Dataset shape: {df.shape}")
print(f"Number of classes: {df['target'].nunique()}")
print(f"\nClass distribution:\n{df['target'].value_counts().sort_index()}")

## 2. Train/Test Split

Split data into training and test sets with stratification.

In [ ]:
X = df.drop(columns=["target"])
y = df["target"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"\nTraining class distribution:\n{pd.Series(y_train).value_counts().sort_index()}")

## 3. Train with Default Settings

Train a classifier with default hyperparameters. DeepTab automatically:
- Detects numerical vs categorical features
- Creates a validation split (20% by default)
- Applies stratified sampling
- Enables early stopping
- Uses GPU if available

In [ ]:
# Instantiate and train
model = MambularClassifier()
model.fit(X_train, y_train, max_epochs=50)

## 4. Evaluate and Predict

In [ ]:
# Evaluate on test set
metrics = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {metrics['accuracy']:.3f}")
print(f"Test Loss: {metrics['loss']:.3f}")

# Get class predictions
predictions = model.predict(X_test)
print(f"\nPredictions shape: {predictions.shape}")
print(f"Sample predictions: {predictions[:10]}")

# Get class probabilities
probabilities = model.predict_proba(X_test)
print(f"\nProbabilities shape: {probabilities.shape}")
print(f"Sample probabilities:\n{probabilities[:3]}")

## 5. Customization with Configs

DeepTab v2.0 uses three independent config classes for fine-grained control:
- **ModelConfig**: Architecture parameters (d_model, n_layers, dropout, etc.)
- **PreprocessingConfig**: Feature engineering (numerical/categorical strategies)
- **TrainerConfig**: Training loop (lr, batch_size, early stopping, etc.)

In [ ]:
from deeptab.configs import MambularConfig, PreprocessingConfig, TrainerConfig

# Model architecture
model_cfg = MambularConfig(
    d_model=128,          # Embedding dimension
    n_layers=6,           # Number of Mamba layers
    dropout=0.3,          # Dropout rate
    use_cls_token=True,   # Classification token
)

# Preprocessing strategy
prep_cfg = PreprocessingConfig(
    numerical_preprocessing="quantile",  # Transform to uniform distribution
    use_ple=True,                         # Piecewise Linear Encoding
    n_bins=50,                            # For binning/PLE
    categorical_preprocessing="ordinal",  # Ordinal encoding
    embedding_dim=16,                     # Categorical embedding dimension
)

# Training loop
trainer_cfg = TrainerConfig(
    lr=1e-3,                          # Learning rate
    batch_size=256,                   # Batch size
    max_epochs=100,                   # Max epochs
    patience=15,                      # Early stopping patience
    lr_scheduler="reduce_on_plateau", # LR scheduling
    optimizer="adamw",                # Optimizer
    weight_decay=1e-4,                # L2 regularization
)

# Create model with custom configs
model_custom = MambularClassifier(
    model_config=model_cfg,
    preprocessing_config=prep_cfg,
    trainer_config=trainer_cfg,
)

# Train
model_custom.fit(X_train, y_train, max_epochs=100)

# Evaluate
metrics_custom = model_custom.evaluate(X_test, y_test)
print(f"Custom Model Accuracy: {metrics_custom['accuracy']:.3f}")

## 6. Hyperparameter Tuning with GridSearchCV

DeepTab models are fully compatible with scikit-learn's GridSearchCV.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define parameter grid
param_grid = {
    "model_config__d_model": [64, 128],
    "model_config__n_layers": [4, 6],
    "trainer_config__lr": [5e-4, 1e-3],
    "preprocessing_config__numerical_preprocessing": ["standard", "quantile"],
}

# Create base model
model_grid = MambularClassifier()

# Grid search with 3-fold CV
grid_search = GridSearchCV(
    model_grid,
    param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=1,  # Use 1 for GPU models
    verbose=2,
)

# Fit (this will take a while)
print("Running grid search...")
grid_search.fit(X_train, y_train)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.3f}")

# Evaluate best model on test set
best_model = grid_search.best_estimator_
test_score = best_model.score(X_test, y_test)
print(f"Test accuracy: {test_score:.3f}")

## 7. Cross-Validation

In [ ]:
from sklearn.model_selection import cross_val_score

model_cv = MambularClassifier()

scores = cross_val_score(
    model_cv, X_train, y_train,
    cv=5,
    scoring="accuracy",
)

print(f"CV scores: {scores}")
print(f"Mean accuracy: {scores.mean():.3f} (+/- {scores.std():.3f})")

## 8. Mixed Data Types (Numerical + Categorical)

DeepTab automatically handles mixed feature types.

In [ ]:
# Create dataset with both numerical and categorical features
df_mixed = pd.DataFrame({
    "age": np.random.randint(18, 80, size=1000),
    "income": np.random.randint(20000, 200000, size=1000),
    "city": np.random.choice(["NYC", "LA", "Chicago"], size=1000),
    "education": np.random.choice(["HS", "BS", "MS", "PhD"], size=1000),
    "target": np.random.randint(0, 2, size=1000),
})

X_mixed = df_mixed.drop(columns=["target"])
y_mixed = df_mixed["target"].values

X_train_mixed, X_test_mixed, y_train_mixed, y_test_mixed = train_test_split(
    X_mixed, y_mixed, test_size=0.2, stratify=y_mixed, random_state=42
)

# Train - automatically detects numerical (age, income) and categorical (city, education)
model_mixed = MambularClassifier()
model_mixed.fit(X_train_mixed, y_train_mixed, max_epochs=50)

metrics_mixed = model_mixed.evaluate(X_test_mixed, y_test_mixed)
print(f"Mixed data accuracy: {metrics_mixed['accuracy']:.3f}")

## 9. Comparing Different Architectures

All DeepTab classifiers share the same API - just change the class name.

In [ ]:
from deeptab.models import (
    FTTransformerClassifier,
    ResNetClassifier,
    NODEClassifier,
    MambularClassifier,
)

# Compare multiple architectures
architectures = [
    FTTransformerClassifier,
    ResNetClassifier,
    NODEClassifier,
    MambularClassifier,
]

results = []
for ModelClass in architectures:
    print(f"\nTraining {ModelClass.__name__}...")
    model = ModelClass()
    model.fit(X_train, y_train, max_epochs=50)
    accuracy = model.score(X_test, y_test)
    results.append((ModelClass.__name__, accuracy))
    print(f"  Accuracy: {accuracy:.3f}")

# Display results
print("\n" + "="*50)
print("Architecture Comparison")
print("="*50)
for name, acc in sorted(results, key=lambda x: x[1], reverse=True):
    print(f"{name:30s}: {acc:.3f}")

## 10. Save and Load Models

In [ ]:
# Save trained model
model.save("classifier_model.pkl")
print("Model saved!")

# Load model
from deeptab.models import MambularClassifier
loaded_model = MambularClassifier.load("classifier_model.pkl")

# Use loaded model
predictions_loaded = loaded_model.predict(X_test)
print(f"Loaded model predictions: {predictions_loaded[:5]}")

## Summary

In this tutorial, you learned how to:
- ✅ Train classification models with DeepTab v2.0
- ✅ Customize architecture, preprocessing, and training with configs
- ✅ Perform hyperparameter tuning with GridSearchCV
- ✅ Handle mixed data types (numerical + categorical)
- ✅ Compare different model architectures
- ✅ Save and load trained models

**Next steps:**
- Try the [Regression Tutorial](regression.ipynb) for continuous targets
- Explore [Distributional Regression](distributional.ipynb) for uncertainty quantification
- Check [Experimental Models](experimental.ipynb) for cutting-edge architectures

**Documentation:** https://deeptab.readthedocs.io/